# 05. Collaborative Filtering

## Objective

The goal of this notebook is to build a personalized movie recommendation system using user-rating behavior.

We will:

* Load the processed ratings dataset
* Create a sparse user-movie rating matrix
* Learn relationships between users and movies
* Generate personalized movie recommendations
* Test the recommendations for selected users

This model will complement the content-based recommender by using actual user preferences rather than movie metadata.


In [6]:
# Import the libraries needed for collaborative filtering
import pandas as pd
import numpy as np

from pathlib import Path

In [7]:
# Define the path to the processed collaborative-filtering dataset
PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

print("Processed data:", PROCESSED_DIR)

Processed data: c:\Users\moham\Desktop\Netflix Movie recomendation system\data\processed


In [8]:
# Load the filtered ratings dataset for collaborative filtering
ratings_cf = pd.read_csv(
    PROCESSED_DIR / "ratings_cf.csv"
)

print("Dataset loaded successfully.")
print("Shape:", ratings_cf.shape)

Dataset loaded successfully.
Shape: (29542223, 4)


In [9]:
# Inspect the structure and first few records of the ratings dataset
print(ratings_cf.columns.tolist())
ratings_cf.head()

['userId', 'movieId', 'rating', 'timestamp']


,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [10]:
# Count the unique users and movies available for collaborative filtering
print("Unique users:", ratings_cf["userId"].nunique())
print("Unique movies:", ratings_cf["movieId"].nunique())
print("Rating range:", ratings_cf["rating"].min(), "to", ratings_cf["rating"].max())

Unique users: 128255
Unique movies: 31701
Rating range: 0.5 to 5.0


In [11]:
# Import the sparse matrix structure used to store the user-movie ratings efficiently
from scipy.sparse import csr_matrix

In [12]:
# Create continuous matrix indices for users and movies
user_ids = ratings_cf["userId"].unique()
movie_ids = ratings_cf["movieId"].unique()

user_to_index = {
    user_id: index
    for index, user_id in enumerate(user_ids)
}

movie_to_index = {
    movie_id: index
    for index, movie_id in enumerate(movie_ids)
}

print("User indices:", len(user_to_index))
print("Movie indices:", len(movie_to_index))

User indices: 128255
Movie indices: 31701


In [13]:
# Convert user and movie IDs into row and column positions for the sparse matrix
user_indices = ratings_cf["userId"].map(user_to_index).to_numpy()
movie_indices = ratings_cf["movieId"].map(movie_to_index).to_numpy()
ratings_values = ratings_cf["rating"].to_numpy()

print("User index array:", user_indices.shape)
print("Movie index array:", movie_indices.shape)
print("Rating value array:", ratings_values.shape)

User index array: (29542223,)
Movie index array: (29542223,)
Rating value array: (29542223,)


In [14]:
# Build the sparse user-movie rating matrix
user_movie_matrix = csr_matrix(
    (
        ratings_values,
        (user_indices, movie_indices)
    ),
    shape=(len(user_ids), len(movie_ids))
)

print("Matrix shape:", user_movie_matrix.shape)
print("Stored ratings:", user_movie_matrix.nnz)

Matrix shape: (128255, 31701)
Stored ratings: 29542223


In [15]:
# Measure how sparse the user-movie rating matrix is
total_possible_ratings = (
    user_movie_matrix.shape[0] * user_movie_matrix.shape[1]
)

actual_ratings = user_movie_matrix.nnz

sparsity = 1 - (actual_ratings / total_possible_ratings)

print("Total possible entries:", total_possible_ratings)
print("Actual ratings:", actual_ratings)
print("Sparsity:", round(sparsity * 100, 2), "%")

Total possible entries: 4065811755
Actual ratings: 29542223
Sparsity: 99.27 %


## Build the collaborative filtering model
In the first model, I'll use Truncated SVD (Singular Value Decomposition). It will learn hidden patterns such as: **users who tend to like certain types of movies may also like other movies with similar rating patterns.**

In [16]:
# Import Truncated SVD for learning hidden patterns in user-movie ratings
from sklearn.decomposition import TruncatedSVD

In [17]:
# Configure the SVD model with 50 hidden factors
svd = TruncatedSVD(
    n_components=50,
    random_state=42
)

print("SVD model configured.")

SVD model configured.


In [39]:
# Train the SVD model on the sparse user-movie rating matrix
user_factors = svd.fit_transform(user_movie_matrix)

print("User factor matrix shape:", user_factors.shape)
print("Explained variance ratio:", round(svd.explained_variance_ratio_.sum(), 4))

User factor matrix shape: (128255, 50)
Explained variance ratio: 0.3478


In [40]:
# Save the learned user factors for use in the Streamlit application
np.save(
    PROJECT_DIR / "models" / "user_factors.npy",
    user_factors.astype(np.float32)
)

print("Saved user_factors.npy")
print("Shape:", user_factors.shape)

Saved user_factors.npy
Shape: (128255, 50)


In [19]:
# Extract the learned latent representation of each movie
movie_factors = svd.components_.T

print("Movie factor matrix shape:", movie_factors.shape)

Movie factor matrix shape: (31701, 50)


In [20]:
# Find the matrix position of User 1 and inspect their learned factors
test_user_id = 1

test_user_index = user_to_index[test_user_id]
test_user_factors = user_factors[test_user_index]

print("User ID:", test_user_id)
print("Matrix index:", test_user_index)
print("Number of latent factors:", len(test_user_factors))

User ID: 1
Matrix index: 0
Number of latent factors: 50


In [21]:
# Calculate a predicted preference score for every movie for User 1
user_scores = test_user_factors @ movie_factors.T

print("Number of movie scores:", len(user_scores))
print("Highest score:", round(user_scores.max(), 4))
print("Lowest score:", round(user_scores.min(), 4))

Number of movie scores: 31701
Highest score: 3.4253
Lowest score: -0.6931


In [22]:
# Find the movies that User 1 has already rated
user_rated_movies = ratings_cf.loc[
    ratings_cf["userId"] == test_user_id,
    "movieId"
].values

print("Movies already rated by User 1:", len(user_rated_movies))
print("First 10 rated movie IDs:", user_rated_movies[:10])

Movies already rated by User 1: 141
First 10 rated movie IDs: [ 17  25  29  30  32  34  36  80 110 111]


In [23]:
# Remove movies that User 1 has already rated from the recommendation candidates
rated_movie_indices = [
    movie_to_index[movie_id]
    for movie_id in user_rated_movies
]

candidate_scores = user_scores.copy()
candidate_scores[rated_movie_indices] = -np.inf

print("Candidate movies:", np.isfinite(candidate_scores).sum())

Candidate movies: 31560


In [24]:
# Select the 10 highest-scoring unseen movies for User 1
top_movie_indices = np.argsort(candidate_scores)[::-1][:10]

recommended_movie_ids = [
    movie_ids[index]
    for index in top_movie_indices
]

print("Recommended movie IDs:", recommended_movie_ids)

Recommended movie IDs: [np.int64(858), np.int64(1193), np.int64(924), np.int64(750), np.int64(1704), np.int64(1617), np.int64(1252), np.int64(1198), np.int64(2858), np.int64(1222)]


In [25]:
# Load movie metadata so recommendation IDs can be converted into movie titles
movies_content = pd.read_csv(
    PROCESSED_DIR / "movies_content.csv"
)

print("Movie metadata loaded:", movies_content.shape)

Movie metadata loaded: (87585, 6)


In [26]:
# Create a readable table of User 1's top collaborative-filtering recommendations
recommendations_cf = movies_content[
    movies_content["movieId"].isin(recommended_movie_ids)
][["movieId", "title"]].copy()

recommendations_cf["score"] = recommendations_cf["movieId"].map(
    dict(zip(recommended_movie_ids, user_scores[top_movie_indices]))
)

recommendations_cf = recommendations_cf.set_index("movieId").loc[
    recommended_movie_ids
].reset_index()

recommendations_cf

,movieId,title,score
0,858,"Godfather, The (1972)",2.582263
1,1193,One Flew Over the Cuckoo's Nest (1975),2.333254
2,924,2001: A Space Odyssey (1968),2.328384
3,750,Dr. Strangelove or: How I Learned to Stop Worr...,2.272384
4,1704,Good Will Hunting (1997),2.070919
5,1617,L.A. Confidential (1997),2.026426
6,1252,Chinatown (1974),1.998941
7,1198,Raiders of the Lost Ark (Indiana Jones and the...,1.981034
8,2858,American Beauty (1999),1.925074
9,1222,Full Metal Jacket (1987),1.861264


In [27]:
# Show the movies that User 1 rated most highly
user_1_ratings = ratings_cf[
    ratings_cf["userId"] == test_user_id
].sort_values("rating", ascending=False)

user_1_top_ratings = user_1_ratings.merge(
    movies_content[["movieId", "title"]],
    on="movieId",
    how="left"
)

user_1_top_ratings[
    ["movieId", "title", "rating"]
].head(10)

,movieId,title,rating
0,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),5.0
1,30,Shanghai Triad (Yao a yao yao dao waipo qiao) ...,5.0
2,166,"Doom Generation, The (1995)",5.0
3,111,Taxi Driver (1976),5.0
4,80,"White Balloon, The (Badkonake sefid) (1995)",5.0
5,1056,Jude (1996),5.0
6,1041,Secrets & Lies (1996),5.0
7,562,Welcome to the Dollhouse (1995),5.0
8,541,Blade Runner (1982),5.0
9,306,Three Colors: Red (Trois couleurs: Rouge) (1994),5.0


In [28]:
# Create a helper function to find the matrix index for a user ID
def get_user_index(user_id):
    return user_to_index.get(user_id)

print("User 1 index:", get_user_index(1))

User 1 index: 0


In [42]:
# Create a reusable function that generates personalized movie recommendations
def recommend_for_user(user_id, n=3):

    # Find the user's matrix position
    user_index = get_user_index(user_id)

    # Check whether the user exists in the collaborative-filtering model
    if user_index is None:
        return "User ID not found in the collaborative-filtering dataset."

    # Calculate preference scores for all movies
    user_scores = user_factors[user_index] @ movie_factors.T

    # Find movies the user has already rated
    user_rated_movies = ratings_cf.loc[
        ratings_cf["userId"] == user_id,
        "movieId"
    ].values

    # Convert rated movie IDs into matrix positions
    rated_movie_indices = [
        movie_to_index[movie_id]
        for movie_id in user_rated_movies
        if movie_id in movie_to_index
    ]

    # Remove already-rated movies from recommendation candidates
    candidate_scores = user_scores.copy()
    candidate_scores[rated_movie_indices] = -np.inf

    # Select the highest-scoring unseen movies
    top_movie_indices = np.argsort(candidate_scores)[::-1][:n]

    # Convert matrix positions back to movie IDs
    recommended_movie_ids = [
        movie_ids[index]
        for index in top_movie_indices
    ]

    # Create a readable recommendation table
    recommendations = movies_content[
        movies_content["movieId"].isin(recommended_movie_ids)
    ][["movieId", "title"]].copy()

    # Add the model's ranking score
    score_map = dict(
        zip(
            recommended_movie_ids,
            user_scores[top_movie_indices]
        )
    )

    recommendations["score"] = recommendations["movieId"].map(score_map)

    # Preserve the recommendation order
    recommendations = (
        recommendations
        .set_index("movieId")
        .loc[recommended_movie_ids]
        .reset_index()
    )

    return recommendations

In [43]:
# Test the personalized recommendation function for User 1
recommend_for_user(1, 10)

,movieId,title,score
0,858,"Godfather, The (1972)",2.582263
1,1193,One Flew Over the Cuckoo's Nest (1975),2.333254
2,924,2001: A Space Odyssey (1968),2.328384
3,750,Dr. Strangelove or: How I Learned to Stop Worr...,2.272384
4,1704,Good Will Hunting (1997),2.070919
5,1617,L.A. Confidential (1997),2.026426
6,1252,Chinatown (1974),1.998941
7,1198,Raiders of the Lost Ark (Indiana Jones and the...,1.981034
8,2858,American Beauty (1999),1.925074
9,1222,Full Metal Jacket (1987),1.861264


In [44]:
# Test personalized recommendations for another user
recommend_for_user(100, 3)

,movieId,title,score
0,1258,"Shining, The (1980)",3.123886
1,2571,"Matrix, The (1999)",3.018271
2,1704,Good Will Hunting (1997),2.833853


In [32]:
# Test how the recommendation function handles an unknown user ID
recommend_for_user(9999999, 10)

'User ID not found in the collaborative-filtering dataset.'

In [46]:
# Create the model folder and import joblib for saving the trained model
import joblib

MODELS_DIR = PROJECT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Models folder:", MODELS_DIR)

Models folder: c:\Users\moham\Desktop\Netflix Movie recomendation system\models


In [47]:
# Save the trained SVD collaborative-filtering model
joblib.dump(
    svd,
    MODELS_DIR / "collaborative_svd.joblib"
)

print("Collaborative filtering model saved successfully.")

Collaborative filtering model saved successfully.


In [48]:
# Save the user and movie ID mappings needed to use the model later
joblib.dump(
    user_to_index,
    MODELS_DIR / "user_to_index.joblib"
)

joblib.dump(
    movie_to_index,
    MODELS_DIR / "movie_to_index.joblib"
)

print("User and movie mappings saved successfully.")

User and movie mappings saved successfully.


In [49]:
# Save the ordered user and movie IDs used by the model's matrix positions
np.save(
    MODELS_DIR / "user_ids.npy",
    user_ids
)

np.save(
    MODELS_DIR / "movie_ids.npy",
    movie_ids
)

print("User and movie ID arrays saved successfully.")

User and movie ID arrays saved successfully.


### Verify that the saved model can be reloaded

In [50]:
# Reload the saved SVD model and verify that it can be used
svd_loaded = joblib.load(
    MODELS_DIR / "collaborative_svd.joblib"
)

print("Loaded model type:", type(svd_loaded).__name__)
print("Number of latent factors:", svd_loaded.n_components)

Loaded model type: TruncatedSVD
Number of latent factors: 50


In [51]:
# Use the reloaded model to recreate the user and movie latent factors
user_factors_loaded = svd_loaded.transform(user_movie_matrix)
movie_factors_loaded = svd_loaded.components_.T

print("Loaded user factors:", user_factors_loaded.shape)
print("Loaded movie factors:", movie_factors_loaded.shape)

Loaded user factors: (128255, 50)
Loaded movie factors: (31701, 50)


## Conclusion

The collaborative-filtering recommender was successfully developed using the MovieLens 32M rating data.

A sparse user-movie rating matrix was created and TruncatedSVD with 50 latent factors was used to learn hidden relationships between users and movies.

The recommender can:

* Generate personalized movie recommendations for a specific user
* Exclude movies the user has already rated
* Produce different recommendations for different users
* Handle invalid user IDs safely
* Save and reload the trained model
* Save the user and movie mappings required for future use

The model's recommendation scores are latent preference scores used for ranking. They should not be interpreted directly as predicted ratings on the 0.5–5.0 MovieLens scale.

The trained collaborative-filtering model and its supporting mappings have been saved in the `models/` directory.

### Next Step

The next notebook will develop an **also-watched recommender** using movie co-occurrence patterns from users' rating histories.
